# Dataset management 02: Explore annotated MSI datasets

This notebook shows how to inspect public MSI dataset metadata with `DatasetExplorer`.

The examples use [METASPACE](https://metaspace2020.eu/). The same `DatasetExplorer` interface can also be initialized for [PRIDE Archive](https://www.ebi.ac.uk/pride/archive/), but each source defines its own filters and metadata fields.

The notebook covers:

1. [creating a dataset explorer](#12-create-datasetexplorer),
2. [checking supported filters and their available values](#2-available-filters),
3. [filtering datasets](#3-dataset-filtering),
4. [reviewing results and rejection reasons](#33-refine-the-filters),
5. [exporting the selected filters to JSON](#4-export-filters).

No `.imzML` or `.ibd` files are downloaded.

### External documentation

- [METASPACE dataset browser](https://metaspace2020.eu/)
- [PRIDE Archive](https://www.ebi.ac.uk/pride/archive/)
- [PRIDE search tutorial](https://www.ebi.ac.uk/training/online/courses/pride-quick-tour/searching-pride/)

<!-- TODO: Add a hyperlink to the tutorial explaining how to run an exported dataset configuration. -->

## 1. Initialization

### 1.1. Set the repository root

The notebook uses paths relative to the project repository. The following cell finds the nearest parent directory containing `pyproject.toml` and changes the working directory to it.


In [1]:
import os
from pathlib import Path

current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").is_file()
)
os.chdir(repository_root)

repository_root


PosixPath('/home/maxi7524/repositories/MSIAutoEncoderWrapper')

### 1.2. Create `DatasetExplorer`

`DatasetExplorer` stores the current filters, search results and manually excluded dataset IDs. It sends source-specific operations to the selected dataset source.

The `source` argument selects the database, currently implemented are:
- `"metaspace"` which uses METASPACE;
- `"pride"` which uses PRIDE.


> Remark: 
>
> For METASPACE, `cache_dir` specifies a directory in which the dataset catalogue is additionally saved as `available-datasets.json`. Later sessions can load this local file instead of requesting the catalogue again.
> - When `cache_dir=None`, the catalogue is kept only in memory.
> - Set `refresh_cache=True` to request the catalogue again and replace the local file. This operation retrieves metadata only.


In [2]:
from IPython.display import display

from msi_autoencoder_wrapper.dataset_management.exploration import DatasetExplorer


explorer = DatasetExplorer(
    source="metaspace",
    # Save the METASPACE dataset catalogue in this directory.
    cache_dir="assets/local/datasets/metaspace",
    # Set to True to request the catalogue again and replace the local file.
    refresh_cache=False,
)

# The same interface can be initialized for PRIDE:
# pride_explorer = DatasetExplorer(source="pride")


2026-08-05 23:16:39,322 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 5 implementation module(s) in package 'msi_autoencoder_wrapper.dataset_management.sources.strategies'.
2026-08-05 23:16:40,744 | INFO     | msi_autoencoder_wrapper.dataset_management.sources.strategies.metaspace:276 | Loaded 19674 METASPACE catalogue records from assets/local/datasets/metaspace/available-datasets.json


## 2. Available filters

Before filtering datasets, we need to inspect:

1. which filter keys are supported;
2. which values occur in the database.

### 2.1. Inspect filter definitions

`get_available_filters()` returns the filter schema for the selected source. For each filter, the schema **may** contain its expected type, default value, provider API field, predefined choices and information about whether it is applied locally.

Filters mapped to a METASPACE API field are sent to METASPACE. Filters marked as local are applied after the datasets have been returned.


In [3]:
available_filters = explorer.get_available_filters()
available_filters


{'name': {'type': 'string', 'api_field': 'nameMask'},
 'dataset_ids': {'type': 'string | list[string]', 'api_field': 'idMask'},
 'submitter_id': {'type': 'string', 'api_field': 'submitter_id'},
 'group_id': {'type': 'string', 'api_field': 'group_id'},
 'project_id': {'type': 'string', 'api_field': 'project_id'},
 'polarity': {'type': 'Positive | Negative',
  'api_field': 'polarity',
  'choices': ['Positive', 'Negative']},
 'analyzer_type': {'type': 'string', 'api_field': 'analyzer_type'},
 'ionisation_source': {'type': 'string', 'api_field': 'ionisation_source'},
 'maldi_matrix': {'type': 'string', 'api_field': 'maldi_matrix'},
 'status': {'type': 'string', 'api_field': 'status', 'default': 'FINISHED'},
 'molecule': {'type': 'string',
  'api_field': 'hasAnnotationMatching.compoundQuery'},
 'annotation_fdr': {'type': 'float', 'default': 0.1, 'local': True},
 'min_annotation_count': {'type': 'integer | null', 'local': True},
 'max_annotation_count': {'type': 'integer | null', 'local': Tr

### 2.2. Inspect available values

`get_available_values(filter_key)` lists values currently present in accessible METASPACE datasets.

The returned table contains:

- `value`: the value used in a filter (yes, this serves as parameter ...);
- `label`: the displayed label;
- `count`: the number of datasets containing that value.

Use this method before entering values such as `organism_part`, `condition` or `ionisation_source`. These fields may use labels different from the label expected by the user.

Only filters with a finite set of values can be inspected this way. Free-text filters such as `name` and numerical filters such as `min_annotation_count` do not provide a value table.


In [4]:
# Replace the key with another enumerable entry from `available_filters`.
organism_values = explorer.get_available_values("organism_part")
display(organism_values.head(30))

# Typical additional inspections:
# display(explorer.get_available_values("organism_part").head(30))
# display(explorer.get_available_values("polarity"))
# display(explorer.get_available_values("analyzer_type").head(30))
# display(explorer.get_available_values("ionisation_source").head(30))
# display(explorer.get_|available_values("maldi_matrix").head(30))


,value,label,count,variants
0,Kidney,Kidney,4393,"Kidney (4035), kidney (342), Kidney (14), kid..."
1,Brain,Brain,2136,"Brain (2101), brain (35)"
2,Cell Line,Cell Line,970,Cell Line (970)
3,Liver,Liver,933,"Liver (875), liver (51), LIVER (7)"
4,Root,Root,870,"Root (457), root (413)"
5,Lung,Lung,810,"Lung (701), lung (109)"
6,Whole organism,Whole organism,807,"Whole organism (772), whole organism (35)"
7,leaf,leaf,784,"leaf (622), Leaf (162)"
8,Breast,Breast,513,Breast (513)
9,N/A,N/A,425,"N/A (423), n/a (2)"


### 2.3. Search within available values

`get_available_values()` returns a pandas `DataFrame`. Use standard pandas filtering to find labels containing a selected phrase.

The following example searches values available for `organism_part`. It does not filter datasets.


In [5]:
organism_part_values = explorer.get_available_values("organism_part")

search_term = "kidney"
matching_organism_parts = organism_part_values[
    organism_part_values["label"].str.contains(
        search_term,
        case=False,
        na=False,
        regex=False,
    )
]

display(matching_organism_parts.head(30))


,value,label,count,variants
0,Kidney,Kidney,4393,"Kidney (4035), kidney (342), Kidney (14), kid..."
10,Kidney Cortex,Kidney Cortex,387,"Kidney Cortex (386), Kidney cortex (1)"
72,Kidney Organoid,Kidney Organoid,25,Kidney Organoid (25)
109,Brain | Lung | Liver | Heart | Kidney | Muscle,Brain | Lung | Liver | Heart | Kidney | Muscle,13,Brain | Lung | Liver | Heart | Kidney | Muscle...
121,Kidney | Muscle,Kidney | Muscle,10,Kidney | Muscle (10)
137,"Liver, kidney, Brain, Spleen","Liver, kidney, Brain, Spleen",8,"Liver, kidney, Brain, Spleen (8)"
235,kidney biopsy,kidney biopsy,2,kidney biopsy (2)
292,Kidney and Liver,Kidney and Liver,1,Kidney and Liver (1)
296,Liver and kidney,Liver and kidney,1,Liver and kidney (1)


## 3. Dataset filtering

### 3.1. Define filters

Pass a dictionary of filter names and values to `explorer.filter(...)`.

Start with a small number of filters. After reviewing the returned datasets, add further conditions.

The example below uses:

- `organism` and `organism_part` to select biological material;
- optional acquisition filters;
- optional annotation-count filters;
- optional molecular statistics;
- `exclude_dataset_ids` for dataset IDs that should not be included.

#### Additional molecular statistics (important)

By default, the result contains annotation counts but not molecule-level statistics.

Set `include_molecule_stats=True` to calculate:

- `molecule_count`: the number of distinct formula–adduct pairs in a dataset;
- `unique_molecule_count`: the number of formula–adduct pairs found only in that dataset within the current result set;
- `unique_molecules`: labels of these pairs.

`min_molecule_count` and `min_unique_molecule_count` use these calculated values as filters.

This requires retrieving annotation identities and therefore performs more API work than filtering only by catalogue metadata. (It takes some time to obtain all of them)

> It still does not download ion images, `.imzML` files or `.ibd` files.


In [38]:
broad_filters = {
    # Biological metadata
    "organism": "Mouse",
    "organism_part": "kidney",
    "condition": "Wildtype",

    # Acquisition metadata
    "polarity": "Negative",
    # "ionisation_source": "MALDI",

    # Annotation filters
    # "status": "FINISHED",
    "annotation_fdr": 0.1,
    # "has_optical_image": True,
    "min_annotation_count": 1,

    # Additional molecular statistics
    # "include_molecule_stats": True,
    # "include_spatial_annotation_stats": True, # REMARK: it filters unique values based on `annotation_fdr`
    # "min_molecule_count": 100,
    # "min_unique_molecule_count": 1,

    # Dataset IDs excluded from the selection
    "exclude_dataset_ids": [],
}

results = explorer.filter(broad_filters)

display(results)
print(f"Found {len(results)} datasets")


2026-08-05 23:53:45,448 | INFO     | msi_autoencoder_wrapper.dataset_management.sources.strategies.metaspace:533 | METASPACE cache reduced discovery to 61 candidate datasets


Query METASPACE dataset catalogue:   0%|          | 0/1 [00:00<?, ?operation/s]

METASPACE discovery:  33%|###3      | 1/3 [00:00<?, ?operation/s]

Retrieve metadata and m/z ranges:   0%|          | 0/1 [00:00<?, ?operation/s]

Retrieve annotation counts:   0%|          | 0/1 [00:00<?, ?operation/s]

2026-08-05 23:53:46,391 | INFO     | msi_autoencoder_wrapper.dataset_management.sources.strategies.metaspace:256 | METASPACE discovery accepted 60 datasets and rejected 8659 datasets
2026-08-05 23:53:46,394 | INFO     | msi_autoencoder_wrapper.dataset_management.exploration.dataset_explorer:120 | Explorer retained 60 records from source metaspace


,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-04-22_21h03m00s,kidney_test_metabolites_null_mz_shift_10_til_550,metaspace,None,https://metaspace2020.eu/dataset/2026-04-22_21...,Mus musculus (mouse),Kidney,Wildtype,Caged,,...,None,None,0.1,None,None,None,None,None,,False
1,2025-04-14_15h53m53s,00070_lgruber_qcl-msi_data11_glomeruli_wt_rms,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_15...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
2,2025-04-14_09h06m15s,00070_lgruber_qcl-msi_het-0717_data3_slide1-rms,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_09...,Mus musculus (mouse),Kidney,Wild type,,,...,None,None,0.1,None,None,None,None,None,,False
3,2025-04-14_09h00m43s,00070_lgruber_qcl-msi_het-0707_data5_slide1-rms,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_09...,Mus musculus (mouse),Kidney,Wild type,,,...,None,None,0.1,None,None,None,None,None,,False
4,2025-04-14_15h55m50s,00070_lgruber_qcl-msi_timson_glomeruli_data2_w...,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_15...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
5,2025-04-14_09h56m19s,00070_lgruber_qcl-msi_wt-4625_data6_slide1-rms,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_09...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
6,2025-04-14_10h18m43s,00070_lgruber_qcl-msi_wt-4628_data1_slide1-rms,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_10...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
7,2025-07-07_13h56m06s,mo6_metabolites,metaspace,None,https://metaspace2020.eu/dataset/2025-07-07_13...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
8,2025-07-07_13h03m27s,msi2024013_20240909_multiomicsiii_nor_300-1350...,metaspace,None,https://metaspace2020.eu/dataset/2025-07-07_13...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
9,2025-07-07_10h55m54s,msi2024013_20240909_multiomicsii_nedc 50-650_f...,metaspace,None,https://metaspace2020.eu/dataset/2025-07-07_10...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False


Found 60 datasets


In [39]:
import pandas as pd

lower_bounds = [100 * i for i in range(1, 5)]
upper_bounds = [100 * i for i in range(7, 21)]

# Ensure numeric comparison
results["mz_min"] = pd.to_numeric(results["mz_min"], errors="coerce")
results["mz_max"] = pd.to_numeric(results["mz_max"], errors="coerce")

range_counts = []

for lower_bound in lower_bounds:
    for upper_bound in upper_bounds:
        if lower_bound >= upper_bound:
            continue

        mask = (
            results["mz_min"].le(lower_bound)
            & results["mz_max"].ge(upper_bound)
        )

        range_counts.append(
            {
                "lower_bound": lower_bound,
                "upper_bound": upper_bound,
                "range_width": upper_bound - lower_bound,
                "dataset_count": int(mask.sum()),
            }
        )

range_counts_df = pd.DataFrame(range_counts)

range_counts_matrix = range_counts_df.pivot(
    index="lower_bound",
    columns="upper_bound",
    values="dataset_count",
)

display(range_counts_matrix)

upper_bound,700,800,900,1000,1100,1200,1300,1400,1500,1600,1700,1800,1900,2000
lower_bound,,,,,,,,,,,,,,
100,15,14,5,3,3,3,0,0,0,0,0,0,0,0
200,40,39,30,8,8,3,0,0,0,0,0,0,0,0
300,42,41,32,10,8,3,0,0,0,0,0,0,0,0
400,50,49,40,18,16,11,8,6,6,6,6,6,6,2


In [40]:
# Categorical summaries
categorical_columns = [
    "organism_parts",
    "condition",
    "polarity",
    "processing_status",
    "has_optical_image",
    "databases",
]

for column in categorical_columns:
    if column not in results:
        continue
    print(f"\n{column}")
    display(results[column].value_counts(dropna=False).head(20))

# Quantitative summaries
quantitative_columns = [
    "pixel_count",
    "annotation_count",
    "molecule_count",
    "unique_molecule_count",
]

available_quantitative_columns = [
    column
    for column in quantitative_columns
    if column in results and results[column].notna().any()
]

if available_quantitative_columns:
    display(results[available_quantitative_columns].describe().T)
else:
    print("No quantitative summary fields are populated for this query.")



organism_parts


organism_parts
Kidney    59
kidney     1
Name: count, dtype: int64


condition


condition
Wildtype     55
Wild type     5
Name: count, dtype: int64


polarity


...


has_optical_image


has_optical_image
False    60
Name: count, dtype: int64


databases


databases
HMDB v4, CoreMetabolome v3, LipidMaps 2017-12-12, SwissLipids 2018-02-02    23
SwissLipids 2018-02-02, CoreMetabolome v3, HMDB v4                           9
HMDB v4, CoreMetabolome v3, LipidMaps 2017-12-12                             5
HMDB v4, LipidMaps 2017-12-12, SwissLipids 2018-02-02                        4
CoreMetabolome v3, LipidMaps 2017-12-12, HMDB v4                             4
HMDB v4                                                                      3
HMDB v4, HMDB-endogenous v4, CoreMetabolome v3                               3
HMDB v4, LipidMaps 2017-12-12, SwissLipids 2018-02-02, KEGG v1               2
HMDB v4, CoreMetabolome v3                                                   2
SwissLipids 2018-02-02, CoreMetabolome v3                                    2
HMDB v4, LipidMaps 2017-12-12, DrugBank 5.1, SwissLipids 2018-02-02          1
HMDB-endogenous v4, HMDB-cotton v4, HMDB v4                                  1
LIPID_MAPS 2016, HMDB v4                  

,count,mean,std,min,25%,50%,75%,max
pixel_count,60.0,34795.583333,34910.788666,5025.0,8572.75,11258.0,61976.50,184011.0
annotation_count,60.0,153.850000,404.376726,3.0,12.75,19.0,74.25,1796.0


Here we can see that proposals that have sense are between 

In [32]:
min_mz = 200
max_mz = 900

mask = (
    (results["mz_min"] <= min_mz)
    & (results["mz_max"] >= max_mz)
)

matching_datasets = results.loc[mask].copy()
non_matching_datasets = results.loc[~mask].copy()

print(f"Matching datasets: {len(matching_datasets)}")
print(f"Non-matching datasets: {len(non_matching_datasets)}")

display(matching_datasets)

Matching datasets: 143
Non-matching datasets: 194


,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-08-03_04h14m21s,dbm-1760_2_S2_SM_Neg_20260731_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-08-03_04...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
1,2026-08-03_04h15m34s,dbdb-1770_2_S2_SM_Neg_20260731_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-08-03_04...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
2,2026-08-03_04h13m27s,dbm-1758_4_S2_SM_Neg_20260731_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-08-03_04...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
3,2026-08-03_04h16m56s,dbdb-1773_2_S2_SM_Neg_20260731_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-08-03_04...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
4,2026-07-30_20h04m37s,WT130_2_S4-2_SM_Neg_20260707_AQ_ML,metaspace,None,https://metaspace2020.eu/dataset/2026-07-30_20...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
330,2017-05-03_18h41m39s,04272017_M_6_4,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_18...,Mus musculus (mouse),Kidney,Diabetic,N/A,,...,None,None,0.1,None,None,None,None,None,,False
331,2017-05-03_18h42m12s,04272017_M_11_3,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_18...,Mus musculus (mouse),Kidney,Diabetic,N/A,,...,None,None,0.1,None,None,None,None,None,,False
332,2017-05-03_18h42m38s,04272017_M_12_5,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_18...,Mus musculus (mouse),Kidney,Diabetic,N/A,,...,None,None,0.1,None,None,None,None,None,,False
333,2017-05-03_18h42m58s,04272017_M_25_4,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_18...,Mus musculus (mouse),Kidney,Diabetic,N/A,,...,None,None,0.1,None,None,None,None,None,,False


In [36]:
matching_datasets

,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-08-03_04h14m21s,dbm-1760_2_S2_SM_Neg_20260731_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-08-03_04...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
1,2026-08-03_04h15m34s,dbdb-1770_2_S2_SM_Neg_20260731_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-08-03_04...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
2,2026-08-03_04h13m27s,dbm-1758_4_S2_SM_Neg_20260731_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-08-03_04...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
3,2026-08-03_04h16m56s,dbdb-1773_2_S2_SM_Neg_20260731_AQ,metaspace,None,https://metaspace2020.eu/dataset/2026-08-03_04...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
4,2026-07-30_20h04m37s,WT130_2_S4-2_SM_Neg_20260707_AQ_ML,metaspace,None,https://metaspace2020.eu/dataset/2026-07-30_20...,Mus musculus (mouse),Kidney,N/A,,,...,None,None,0.1,None,None,None,None,None,,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
330,2017-05-03_18h41m39s,04272017_M_6_4,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_18...,Mus musculus (mouse),Kidney,Diabetic,N/A,,...,None,None,0.1,None,None,None,None,None,,False
331,2017-05-03_18h42m12s,04272017_M_11_3,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_18...,Mus musculus (mouse),Kidney,Diabetic,N/A,,...,None,None,0.1,None,None,None,None,None,,False
332,2017-05-03_18h42m38s,04272017_M_12_5,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_18...,Mus musculus (mouse),Kidney,Diabetic,N/A,,...,None,None,0.1,None,None,None,None,None,,False
333,2017-05-03_18h42m58s,04272017_M_25_4,metaspace,None,https://metaspace2020.eu/dataset/2017-05-03_18...,Mus musculus (mouse),Kidney,Diabetic,N/A,,...,None,None,0.1,None,None,None,None,None,,False


In [19]:
explorer.get_available_filters()

{'name': {'type': 'string', 'api_field': 'nameMask'},
 'dataset_ids': {'type': 'string | list[string]', 'api_field': 'idMask'},
 'submitter_id': {'type': 'string', 'api_field': 'submitter_id'},
 'group_id': {'type': 'string', 'api_field': 'group_id'},
 'project_id': {'type': 'string', 'api_field': 'project_id'},
 'polarity': {'type': 'Positive | Negative',
  'api_field': 'polarity',
  'choices': ['Positive', 'Negative']},
 'analyzer_type': {'type': 'string', 'api_field': 'analyzer_type'},
 'ionisation_source': {'type': 'string', 'api_field': 'ionisation_source'},
 'maldi_matrix': {'type': 'string', 'api_field': 'maldi_matrix'},
 'status': {'type': 'string', 'api_field': 'status', 'default': 'FINISHED'},
 'molecule': {'type': 'string',
  'api_field': 'hasAnnotationMatching.compoundQuery'},
 'annotation_fdr': {'type': 'float', 'default': 0.1, 'local': True},
 'min_annotation_count': {'type': 'integer | null', 'local': True},
 'max_annotation_count': {'type': 'integer | null', 'local': Tr

In [37]:
filters = {
    # Biological and acquisition metadata
    "organism": "Mouse",
    "organism_part": "Kidney",
    "polarity": "Negative",
    "condition": "Wildtype",  # matched tolerantly; also groups "Wildtype", "wildtype", etc.


    # Annotation filters and molecular statistics
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
    "include_molecule_stats": True,
    "include_spatial_annotation_stats": True, # REMARK: it filters unique values based on `annotation_fdr`

    # Dataset IDs excluded from the selection
    ## i put here datasets that do not match 
    "exclude_dataset_ids": non_matching_datasets['dataset_id'].to_list(),
}

results_kidney = explorer.filter(filters)

display(results_kidney)
print(f"Accepted {len(results_kidney)} datasets") 


2026-08-05 23:52:03,074 | INFO     | msi_autoencoder_wrapper.dataset_management.sources.strategies.metaspace:533 | METASPACE cache reduced discovery to 61 candidate datasets


Query METASPACE dataset catalogue:   0%|          | 0/1 [00:00<?, ?operation/s]

METASPACE discovery:   2%|1         | 1/65 [00:00<?, ?operation/s]

Retrieve metadata and m/z ranges:   0%|          | 0/1 [00:00<?, ?operation/s]

Retrieve annotation counts:   0%|          | 0/1 [00:00<?, ?operation/s]

...

Retrieve 73 ion images for 2017-04-20_09h06m22s:   0%|          | 0/1 [00:00<?, ?operation/s]

Retrieve molecular identities:   0%|          | 0/1 [00:00<?, ?operation/s]

2026-08-05 23:53:44,750 | INFO     | msi_autoencoder_wrapper.dataset_management.sources.strategies.metaspace:256 | METASPACE discovery accepted 60 datasets and rejected 8659 datasets
2026-08-05 23:53:44,753 | INFO     | msi_autoencoder_wrapper.dataset_management.exploration.dataset_explorer:120 | Explorer retained 30 records from source metaspace


,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-04-22_21h03m00s,kidney_test_metabolites_null_mz_shift_10_til_550,metaspace,None,https://metaspace2020.eu/dataset/2026-04-22_21...,Mus musculus (mouse),Kidney,Wildtype,Caged,,...,75,0.991254,0.1,7,1,complete,7,3,"C13H14O3S-H, C24H28O5-H, C24H30O5-H",False
1,2025-07-06_22h07m04s,msi2024013_20240909_multiomicsii_nor 50-1350,metaspace,None,https://metaspace2020.eu/dataset/2025-07-06_22...,Mus musculus (mouse),Kidney,Wildtype,,,...,0,1.000000,0.1,118,3,complete,103,8,"C10H16N4O7S-H, C10H9Cl3O2-H, C3H8O10P2+Cl, C43...",False
2,2025-07-06_16h31m32s,msi2024013_20240909_multiomicsii_nedc 50-1350,metaspace,None,https://metaspace2020.eu/dataset/2025-07-06_16...,Mus musculus (mouse),Kidney,Wildtype,,,...,86,0.998785,0.1,32,2,complete,23,7,"C47H83NO10P-H, C48H84NO10P-H, C49H86NO10P-H, C...",False
3,2024-05-23_14h25m01s,K3 vs K6 neg -Jano,metaspace,None,https://metaspace2020.eu/dataset/2024-05-23_14...,Mouse,Kidney,Wildtype,,,...,67,0.996666,0.1,485,4,complete,311,117,"C10H21N3O[M]-, C10H7N3S[M]-, C11H9F2NO3-H, C12...",False
4,2024-04-10_17h06m19s,8223301,metaspace,None,https://metaspace2020.eu/dataset/2024-04-10_17...,Mus musculus (mouse),Kidney,Wildtype,,,...,0,1.000000,0.1,216,3,complete,124,31,"C10H14N4O5-H, C10H15N3O5+Cl, C11H20O7-H, C12H2...",False
5,2024-02-20_01h55m56s,d28-2014-2,metaspace,None,https://metaspace2020.eu/dataset/2024-02-20_01...,Mus musculus (mouse),Kidney,Wildtype,N/A,,...,4,0.999444,0.1,10,3,complete,7,0,,False
6,2024-02-20_01h57m32s,d28-2017-2,metaspace,None,https://metaspace2020.eu/dataset/2024-02-20_01...,Mus musculus (mouse),Kidney,Wildtype,N/A,,...,6,0.999217,0.1,18,3,complete,14,6,"C44H81O7P-H, C45H79O12P-H, C46H83O12P-H, C49H8...",False
7,2024-02-20_01h54m01s,d14-2295,metaspace,None,https://metaspace2020.eu/dataset/2024-02-20_01...,Mus musculus (mouse),Kidney,Wildtype,N/A,,...,3,0.999403,0.1,14,4,complete,10,6,"C15H19NO9[M]-, C17H12O10S[M]-, C18H12O4[M]-, C...",False
8,2024-02-20_01h54m41s,d28-2006-2,metaspace,None,https://metaspace2020.eu/dataset/2024-02-20_01...,Mus musculus (mouse),Kidney,Wildtype,N/A,,...,0,1.000000,0.1,17,4,complete,14,1,C49H93O12P[M]-,False
9,2024-02-20_01h53m31s,d14-2015,metaspace,None,https://metaspace2020.eu/dataset/2024-02-20_01...,Mus musculus (mouse),Kidney,Wildtype,N/A,,...,0,1.000000,0.1,22,4,complete,14,5,"C14H20N2O3S-H, C43H78NO8P[M]-, C46H79O7P[M]-, ...",False


Accepted 30 datasets


In [ ]:
## to gigabyte transfer 
results_kidney['total_size_bytes'].sum() / 10**(9)

np.float64(30.238378985)

### 3.4. Inspect rejected datasets

`explorer.rejected()` returns datasets rejected by local filters. The table may contain the dataset ID, dataset name, rejection reason and METASPACE URL.

It does not contain datasets removed directly by METASPACE API filters, because such datasets **are not returned to the explorer**.

Use this table to check which local threshold removed each dataset.


In [9]:
rejected = explorer.rejected()

display(rejected)
print(f"Rejected locally: {len(rejected)} datasets")


""


Rejected locally: 0 datasets


### 3.5. Inspect full metadata

The result table contains selected fields in a common format. To inspect the full metadata returned for one dataset, call `get_dataset_metadata(dataset_id)` on the selected source.

The `project_url` column links to the dataset page in METASPACE.


In [46]:
if not results_kidney.empty:
    dataset_id = results_kidney.iloc[0]["dataset_id"]
    dataset_record = explorer.source.get_dataset_metadata(dataset_id)

    display(dataset_record)
    print(results_kidney.iloc[0]["project_url"])
else:
    print("The current query returned no datasets.")


{'dataset_id': '2026-04-22_21h03m00s',
 'name': 'kidney_test_metabolites_null_mz_shift_10_til_550',
 'metadata': {'Data_Type': 'Imaging MS',
  'Sample_Information': {'Organism': 'Mus musculus (mouse)',
   'Organism_Part': 'Kidney',
   'Condition': 'Wildtype',
   'Sample_Growth_Conditions': 'Caged'},
  'Sample_Preparation': {'Sample_Stabilisation': 'None',
   'Tissue_Modification': 'None',
   'MALDI_Matrix': 'n-(1-naphthyl)ethylenediamine dihydrochloride (NEDC)',
   'MALDI_Matrix_Application': 'HTX',
   'Solvent': '70% MeOH'},
  'MS_Analysis': {'Polarity': 'Negative',
   'Ionisation_Source': 'MALDI',
   'Analyzer': 'Orbitrap',
   'Detector_Resolving_Power': {'Resolving_Power': 120000, 'mz': 200},
   'Pixel_Size': {'Xaxis': 50, 'Yaxis': 50}},
  'project_url': 'https://metaspace2020.eu/dataset/2026-04-22_21h03m00s',
  'provider_metadata': {'id': '2026-04-22_21h03m00s',
   'name': 'kidney_test_metabolites_null_mz_shift_10_til_550',
   'uploadDT': '2026-04-22T19:03:01.394Z',
   'submitter':

https://metaspace2020.eu/dataset/2026-04-22_21h03m00s


### 3.6. Exclude reviewed datasets

Use manual exclusions when a dataset passes the filters but should not be used in the experiment.

- `exclude(dataset_ids)` removes IDs from the displayed selection and adds them to the exported exclusion list;
- `include(dataset_ids)` removes IDs from that exclusion list;
- `results(include_excluded=True)` displays accepted and manually excluded datasets together.

An ID can be excluded only after it has appeared in the current search results.


In [48]:
results_kidney

,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-04-22_21h03m00s,kidney_test_metabolites_null_mz_shift_10_til_550,metaspace,None,https://metaspace2020.eu/dataset/2026-04-22_21...,Mus musculus (mouse),Kidney,Wildtype,Caged,,...,75,0.991254,0.1,7,1,complete,7,3,"C13H14O3S-H, C24H28O5-H, C24H30O5-H",False
1,2025-07-06_22h07m04s,msi2024013_20240909_multiomicsii_nor 50-1350,metaspace,None,https://metaspace2020.eu/dataset/2025-07-06_22...,Mus musculus (mouse),Kidney,Wildtype,,,...,0,1.000000,0.1,118,3,complete,103,8,"C10H16N4O7S-H, C10H9Cl3O2-H, C3H8O10P2+Cl, C43...",False
2,2025-07-06_16h31m32s,msi2024013_20240909_multiomicsii_nedc 50-1350,metaspace,None,https://metaspace2020.eu/dataset/2025-07-06_16...,Mus musculus (mouse),Kidney,Wildtype,,,...,86,0.998785,0.1,32,2,complete,23,7,"C47H83NO10P-H, C48H84NO10P-H, C49H86NO10P-H, C...",False
3,2024-05-23_14h25m01s,K3 vs K6 neg -Jano,metaspace,None,https://metaspace2020.eu/dataset/2024-05-23_14...,Mouse,Kidney,Wildtype,,,...,67,0.996666,0.1,485,4,complete,311,117,"C10H21N3O[M]-, C10H7N3S[M]-, C11H9F2NO3-H, C12...",False
4,2024-04-10_17h06m19s,8223301,metaspace,None,https://metaspace2020.eu/dataset/2024-04-10_17...,Mus musculus (mouse),Kidney,Wildtype,,,...,0,1.000000,0.1,216,3,complete,124,31,"C10H14N4O5-H, C10H15N3O5+Cl, C11H20O7-H, C12H2...",False
5,2024-02-20_01h55m56s,d28-2014-2,metaspace,None,https://metaspace2020.eu/dataset/2024-02-20_01...,Mus musculus (mouse),Kidney,Wildtype,N/A,,...,4,0.999444,0.1,10,3,complete,7,0,,False
6,2024-02-20_01h57m32s,d28-2017-2,metaspace,None,https://metaspace2020.eu/dataset/2024-02-20_01...,Mus musculus (mouse),Kidney,Wildtype,N/A,,...,6,0.999217,0.1,18,3,complete,14,6,"C44H81O7P-H, C45H79O12P-H, C46H83O12P-H, C49H8...",False
7,2024-02-20_01h54m01s,d14-2295,metaspace,None,https://metaspace2020.eu/dataset/2024-02-20_01...,Mus musculus (mouse),Kidney,Wildtype,N/A,,...,3,0.999403,0.1,14,4,complete,10,6,"C15H19NO9[M]-, C17H12O10S[M]-, C18H12O4[M]-, C...",False
8,2024-02-20_01h54m41s,d28-2006-2,metaspace,None,https://metaspace2020.eu/dataset/2024-02-20_01...,Mus musculus (mouse),Kidney,Wildtype,N/A,,...,0,1.000000,0.1,17,4,complete,14,1,C49H93O12P[M]-,False
9,2024-02-20_01h53m31s,d14-2015,metaspace,None,https://metaspace2020.eu/dataset/2024-02-20_01...,Mus musculus (mouse),Kidney,Wildtype,N/A,,...,0,1.000000,0.1,22,4,complete,14,5,"C14H20N2O3S-H, C43H78NO8P[M]-, C46H79O7P[M]-, ...",False


In [47]:
excluded_dataset_ids = []

if excluded_dataset_ids:
    explorer.exclude(excluded_dataset_ids)

display(explorer.results())


,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-04-22_21h03m00s,kidney_test_metabolites_null_mz_shift_10_til_550,metaspace,None,https://metaspace2020.eu/dataset/2026-04-22_21...,Mus musculus (mouse),Kidney,Wildtype,Caged,,...,None,None,0.1,None,None,None,None,None,,False
1,2025-04-14_15h53m53s,00070_lgruber_qcl-msi_data11_glomeruli_wt_rms,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_15...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
2,2025-04-14_09h06m15s,00070_lgruber_qcl-msi_het-0717_data3_slide1-rms,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_09...,Mus musculus (mouse),Kidney,Wild type,,,...,None,None,0.1,None,None,None,None,None,,False
3,2025-04-14_09h00m43s,00070_lgruber_qcl-msi_het-0707_data5_slide1-rms,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_09...,Mus musculus (mouse),Kidney,Wild type,,,...,None,None,0.1,None,None,None,None,None,,False
4,2025-04-14_15h55m50s,00070_lgruber_qcl-msi_timson_glomeruli_data2_w...,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_15...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
5,2025-04-14_09h56m19s,00070_lgruber_qcl-msi_wt-4625_data6_slide1-rms,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_09...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
6,2025-04-14_10h18m43s,00070_lgruber_qcl-msi_wt-4628_data1_slide1-rms,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_10...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
7,2025-07-07_13h56m06s,mo6_metabolites,metaspace,None,https://metaspace2020.eu/dataset/2025-07-07_13...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
8,2025-07-07_13h03m27s,msi2024013_20240909_multiomicsiii_nor_300-1350...,metaspace,None,https://metaspace2020.eu/dataset/2025-07-07_13...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
9,2025-07-07_10h55m54s,msi2024013_20240909_multiomicsii_nedc 50-650_f...,metaspace,None,https://metaspace2020.eu/dataset/2025-07-07_10...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False


## 4. Export filters

### 4.1. Write the JSON file

`export_config(path)` writes the current filters to JSON. Dataset IDs added with `exclude(...)` are included in `exclude_dataset_ids`.

The file contains only the dataset search configuration.


In [50]:
explorer.accepted()

,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-04-22_21h03m00s,kidney_test_metabolites_null_mz_shift_10_til_550,metaspace,None,https://metaspace2020.eu/dataset/2026-04-22_21...,Mus musculus (mouse),Kidney,Wildtype,Caged,,...,None,None,0.1,None,None,None,None,None,,False
1,2025-04-14_15h53m53s,00070_lgruber_qcl-msi_data11_glomeruli_wt_rms,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_15...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
2,2025-04-14_09h06m15s,00070_lgruber_qcl-msi_het-0717_data3_slide1-rms,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_09...,Mus musculus (mouse),Kidney,Wild type,,,...,None,None,0.1,None,None,None,None,None,,False
3,2025-04-14_09h00m43s,00070_lgruber_qcl-msi_het-0707_data5_slide1-rms,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_09...,Mus musculus (mouse),Kidney,Wild type,,,...,None,None,0.1,None,None,None,None,None,,False
4,2025-04-14_15h55m50s,00070_lgruber_qcl-msi_timson_glomeruli_data2_w...,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_15...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
5,2025-04-14_09h56m19s,00070_lgruber_qcl-msi_wt-4625_data6_slide1-rms,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_09...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
6,2025-04-14_10h18m43s,00070_lgruber_qcl-msi_wt-4628_data1_slide1-rms,metaspace,None,https://metaspace2020.eu/dataset/2025-04-14_10...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
7,2025-07-07_13h56m06s,mo6_metabolites,metaspace,None,https://metaspace2020.eu/dataset/2025-07-07_13...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
8,2025-07-07_13h03m27s,msi2024013_20240909_multiomicsiii_nor_300-1350...,metaspace,None,https://metaspace2020.eu/dataset/2025-07-07_13...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
9,2025-07-07_10h55m54s,msi2024013_20240909_multiomicsii_nedc 50-650_f...,metaspace,None,https://metaspace2020.eu/dataset/2025-07-07_10...,Mus musculus (mouse),Kidney,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False


In [ ]:
output_path = Path(
    "assets/configs/datasets/metaspace_mouse_liver.json"
)

exported_path = explorer.export_config(output_path)
print(exported_path)


2026-07-31 11:49:16,574 | INFO     | msi_autoencoder_wrapper.dataset_management.exploration.dataset_explorer:188 | Exported dataset query configuration to assets/configs/datasets/metaspace_mouse_liver.json
assets/configs/datasets/metaspace_mouse_liver.json


### 4.2. Check the exported file

Read the JSON file and inspect its content before using it in another command.


In [13]:
import json

exported_config = json.loads(exported_path.read_text(encoding="utf-8"))
exported_config


{'organism': 'Mouse',
 'name': 'liver',
 'organism_part': 'Liver',
 'polarity': 'Negative',
 'condition': 'Wild type',
 'annotation_fdr': 0.1,
 'min_annotation_count': 1,
 'include_molecule_stats': True,
 'include_spatial_annotation_stats': True,
 'exclude_dataset_ids': []}

## 5. Next step

The output of this notebook is the exported JSON file.

<!-- # TODO: Add a hyperlink to the tutorial explaining how to run an exported dataset configuration. -->

Running the query configuration and downloading `.imzML` and `.ibd` files are described in [Dataset management 03](dataset_management_03_metaspace_download_and_merge.ipynb).
